# DPO Fine-tuning — google/gemma-2b
**Acceleratore:** 2× T4 | **Metodo:** QLoRA + DPO | **Dataset:** custom Kaggle dataset

In [1]:
# ── Cella 1 · Installazione dipendenze ──────────────────────────────────────
import subprocess, sys

packages = [
    "transformers>=4.40.0",
    "trl>=0.8.6",
    "peft>=0.10.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.29.0",
    "datasets>=2.19.0",
    "einops",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages
)
print("✅ Dipendenze installate")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.6/663.6 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 66.0 MB/s eta 0:00:00
✅ Dipendenze installate


In [2]:
# ── Cella 2 · Setup cartelle (aggira il filesystem read-only di Kaggle) ─────
import os

# /kaggle/working/ è SEMPRE scrivibile su Kaggle
BASE_DIR   = "/kaggle/working"
OUTPUT_DIR = os.path.join(BASE_DIR, "gemma2b-dpo")
CACHE_DIR  = os.path.join(BASE_DIR, "hf_cache")
LOGS_DIR   = os.path.join(BASE_DIR, "logs")

for d in [OUTPUT_DIR, CACHE_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# Reindirizza la cache di HuggingFace nella cartella scrivibile
os.environ["HF_HOME"]              = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"]   = os.path.join(CACHE_DIR, "transformers")
os.environ["HF_DATASETS_CACHE"]    = os.path.join(CACHE_DIR, "datasets")
# Necessario per Gemma su Kaggle (evita errori di tokenizer)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

print(f"📁 Output  → {OUTPUT_DIR}")
print(f"📁 Cache   → {CACHE_DIR}")

📁 Output  → /kaggle/working/gemma2b-dpo
📁 Cache   → /kaggle/working/hf_cache


In [3]:
# ── Cella 3 · Verifica GPU ───────────────────────────────────────────────────
import torch

print(f"CUDA disponibile : {torch.cuda.is_available()}")
print(f"Numero di GPU    : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram  = props.total_memory / 1024**3
    print(f"  GPU {i}: {props.name} — {vram:.1f} GB VRAM")

CUDA disponibile : True
Numero di GPU    : 2
  GPU 0: Tesla T4 — 14.6 GB VRAM
  GPU 1: Tesla T4 — 14.6 GB VRAM


In [4]:
from datasets import load_from_disk

ds = load_from_disk("/kaggle/input/datasets/lorenzosalis/dataset-dpo-trl")

# Se è un DatasetDict
from datasets import DatasetDict
if isinstance(ds, DatasetDict):
    ds = ds["train"]

ds.to_parquet("/kaggle/working/dataset_dpo.parquet")
print("✅ Salvato in /kaggle/working/dataset_dpo.parquet")

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

✅ Salvato in /kaggle/working/dataset_dpo.parquet


In [5]:
# ── Cella 4 · Configurazione ─────────────────────────────────────────────────

# ⚠️  Modifica DATASET_PATH con il percorso reale del tuo dataset su Kaggle
# Es: /kaggle/input/<nome-dataset>/<file>.csv   oppure  .json / .parquet
DATASET_PATH = "/kaggle/working/dataset_dpo.parquet"

# Colonne del dataset — rinomina se necessario
COL_PROMPT   = "prompt"
COL_CHOSEN   = "chosen"
COL_REJECTED = "rejected"

MODEL_ID     = "google/gemma-2b"

# Iperparametri — ottimizzati per dataset piccolo (<1k) su 2×T4
NUM_EPOCHS        = 3
BATCH_SIZE        = 1      # per GPU; effective batch = 2 × 2 GPU × 4 grad_accum = 16
GRAD_ACCUM        = 8
LEARNING_RATE     = 5e-5
MAX_LENGTH        = 384    # token totali (prompt + risposta)
BETA              = 0.1    # parametro DPO (trade-off KL)
WARMUP_RATIO      = 0.1

# LoRA
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05

print("✅ Configurazione caricata")

✅ Configurazione caricata


In [6]:
# ── Cella 5 · Caricamento dataset ────────────────────────────────────────────
from datasets import load_dataset, DatasetDict

ext = DATASET_PATH.rsplit(".", 1)[-1].lower()
fmt_map = {"json": "json", "jsonl": "json", "csv": "csv", "parquet": "parquet"}
fmt = fmt_map.get(ext, "json")

raw = load_dataset(fmt, data_files=DATASET_PATH, split="train")

# Rinomina colonne se necessario
rename = {}
for target, src in [("prompt", COL_PROMPT),
                    ("chosen", COL_CHOSEN),
                    ("rejected", COL_REJECTED)]:
    if src != target and src in raw.column_names:
        rename[src] = target
if rename:
    raw = raw.rename_columns(rename)

# Mantieni solo le colonne DPO
raw = raw.select_columns(["prompt", "chosen", "rejected"])

# Split train / eval (90/10) — fondamentale con dataset piccolo
split    = raw.train_test_split(test_size=0.1, seed=42)
ds_train = split["train"]
ds_eval  = split["test"]

print(f"Train: {len(ds_train)} esempi")
print(f"Eval : {len(ds_eval)} esempi")
print("\nEsempio:")
print(ds_train[0])

Generating train split: 0 examples [00:00, ? examples/s]

Train: 236 esempi
Eval : 27 esempi

Esempio:
{'prompt': 'Hereâ€™s a highly challenging, multi-disciplinary coding question that requires synthesis of **low-level optimizations, functional programming, concurrency, and domain-specific knowledge** (e.g., parsing, serialization, and probabilistic modeling). Itâ€™s designed to test deep understanding while being concise (under 2000 tokens).\n\n---\n\n### **Question: "The Parallel Parsing Pipeline with Probabilistic Validation"**\n**Languages:** Rust (for low-level control), Haskell (for functional purity), or Python (with asyncio + multiprocessing).\n**Constraints:**\n- No external libraries (except standard libraries).\n- Must handle edge cases explicitly (e.g., malformed input, race conditions).\n- Optimize for both time and space complexity.\n\n---\n\n#### **Problem Statement**\nYou are building a **high-performance, parallelizable log parser** for a distributed telemetry system. Each log line is a JSON-like string with **variable schem

In [7]:
# ── Cella 5b · Autenticazione HuggingFace ────────────────────────────────────
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secret = UserSecretsClient()
hf_token = secret.get_secret("HF_TOKEN")
login(token=hf_token)
print("✅ Autenticato su HuggingFace")

✅ Autenticato su HuggingFace


In [8]:
# ── Cella 6 · Tokenizer ───────────────────────────────────────────────────────
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
)

# Gemma non ha pad_token di default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "right"   # necessario per DPO con causal LM

print(f"Vocab size   : {tokenizer.vocab_size}")
print(f"Pad token    : {tokenizer.pad_token!r}")
print(f"Padding side : {tokenizer.padding_side}")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Vocab size   : 256000
Pad token    : '<pad>'
Padding side : right


In [9]:
# ── Cella 7 · Caricamento modello in 4-bit (QLoRA) ───────────────────────────
import os
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Con DDP ogni processo carica il modello sul proprio device.
# local_rank viene settato da accelerate/torchrun; in un notebook
# siamo sempre nel processo principale → rank 0 → cuda:0.
# Il DPO Trainer con accelerate gestisce lui la replica sulla GPU 1.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},           # ← tutto su cuda:0; accelerate clonerà su cuda:1
    cache_dir=CACHE_DIR,
    trust_remote_code=True,
    dtype=torch.float16,
    attn_implementation="eager",
)

model.config.use_cache = False
model.config.pretraining_tp = 1

from collections import Counter
device_counts = Counter(str(p.device) for p in model.parameters())
print("Distribuzione parametri per device:")
for dev, cnt in device_counts.items():
    print(f"  {dev}: {cnt} tensori")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Distribuzione parametri per device:
  cuda:0: 164 tensori


In [10]:
# ── Cella 8 · Configurazione LoRA ─────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=8,               
    lora_alpha=16,      
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"], 
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 921,600 || all params: 2,507,094,016 || trainable%: 0.0368


In [11]:
# ── Cella 9 · [RIMOSSA] Training arguments ridondanti ────────────────────────
# La cella originale creava un oggetto TrainingArguments che non veniva mai
# usato: il DPO Trainer accetta direttamente DPOConfig (cella 10).
# Lasciata come placeholder per non spostare la numerazione delle celle.
print("ℹ️  Cella 9 non più necessaria — configurazione centralizzata in DPOConfig (cella 10).")


ℹ️  Cella 9 non più necessaria — configurazione centralizzata in DPOConfig (cella 10).


In [12]:
# ── Cella 10 · DPO Trainer e avvio training ───────────────────────────────────
import torch
from trl import DPOTrainer, DPOConfig

torch.cuda.empty_cache()

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=WARMUP_RATIO,
    fp16=False,
    bf16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    beta=BETA,
    max_length=MAX_LENGTH,
    dataloader_pin_memory=False,
    truncation_mode="keep_start",
    loss_type="sigmoid",
    label_smoothing=0.0,
    disable_dropout=True,
    precompute_ref_log_probs=True,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=ds_train,
    eval_dataset=ds_eval,
    processing_class=tokenizer,
)

print("🚀 Avvio training DPO...")
train_result = trainer.train()

print("\n📊 Risultati training:")
print(train_result)

Adding EOS to train dataset:   0%|          | 0/236 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/236 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+chosen. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized 

Adding EOS to eval dataset:   0%|          | 0/27 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/27 [00:00<?, ? examples/s]

...

Computing reference log probs for eval dataset:   0%|          | 0/27 [00:00<?, ?it/s]

🚀 Avvio training DPO...


Epoch,Training Loss,Validation Loss
1,0.690760,0.665659
2,0.649795,0.641578
3,0.636293,0.629816


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]


📊 Risultati training:
TrainOutput(global_step=45, training_loss=0.6588668558332655, metrics={'train_runtime': 4186.3297, 'train_samples_per_second': 0.169, 'train_steps_per_second': 0.011, 'total_flos': 6471839933005824.0, 'train_loss': 0.6588668558332655})


In [13]:
# ── Cella 11 · Salvataggio modello finale ─────────────────────────────────────
import os

FINAL_DIR = os.path.join(OUTPUT_DIR, "final")
os.makedirs(FINAL_DIR, exist_ok=True)

# Salva adapter LoRA
trainer.model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Salva le metriche di training
metrics = train_result.metrics
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

print(f"\n✅ Modello salvato in: {FINAL_DIR}")
print("\nFile salvati:")
for f in os.listdir(FINAL_DIR):
    size = os.path.getsize(os.path.join(FINAL_DIR, f)) / 1024**2
    print(f"  {f:40s} {size:.1f} MB")

***** train metrics *****
  total_flos               =  6027370GF
  train_loss               =     0.6589
  train_runtime            = 1:09:46.32
  train_samples_per_second =      0.169
  train_steps_per_second   =      0.011

✅ Modello salvato in: /kaggle/working/gemma2b-dpo/final

File salvati:
  ref                                      0.0 MB
  adapter_model.safetensors                3.5 MB
  README.md                                0.0 MB
  tokenizer.json                           32.8 MB
  adapter_config.json                      0.0 MB
  tokenizer_config.json                    0.0 MB


In [14]:
# ── Cella 12 · (Opzionale) Merge LoRA → modello completo ─────────────────────
# Esegui questa cella SOLO se vuoi un modello standalone (senza PEFT)
# Richiede più VRAM: potrebbe non entrare in memoria sulle T4.
# Se fallisce, usa direttamente l'adapter salvato sopra.

'''
MERGED_DIR = os.path.join(OUTPUT_DIR, "merged")

try:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    import torch

    print("Caricamento modello base in fp16 per il merge...")
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        cache_dir=CACHE_DIR,
        trust_remote_code=True,
    )

    print("Merge LoRA adapter...")
    merged = PeftModel.from_pretrained(base_model, FINAL_DIR)
    merged = merged.merge_and_unload()

    os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)

    print(f"✅ Modello merged salvato in: {MERGED_DIR}")
    del merged, base_model
    torch.cuda.empty_cache()

except RuntimeError as e:
    print(f"⚠️  Merge non riuscito (VRAM insufficiente): {e}")
    print("Usa direttamente l'adapter LoRA da:", FINAL_DIR)
'''

'\nMERGED_DIR = os.path.join(OUTPUT_DIR, "merged")\n\ntry:\n    from peft import PeftModel\n    from transformers import AutoModelForCausalLM\n    import torch\n\n    print("Caricamento modello base in fp16 per il merge...")\n    base_model = AutoModelForCausalLM.from_pretrained(\n        MODEL_ID,\n        torch_dtype=torch.float16,\n        device_map="auto",\n        cache_dir=CACHE_DIR,\n        trust_remote_code=True,\n    )\n\n    print("Merge LoRA adapter...")\n    merged = PeftModel.from_pretrained(base_model, FINAL_DIR)\n    merged = merged.merge_and_unload()\n\n    os.makedirs(MERGED_DIR, exist_ok=True)\n    merged.save_pretrained(MERGED_DIR, safe_serialization=True)\n    tokenizer.save_pretrained(MERGED_DIR)\n\n    print(f"✅ Modello merged salvato in: {MERGED_DIR}")\n    del merged, base_model\n    torch.cuda.empty_cache()\n\nexcept RuntimeError as e:\n    print(f"⚠️  Merge non riuscito (VRAM insufficiente): {e}")\n    print("Usa direttamente l\'adapter LoRA da:", FINAL_DIR)

In [15]:
'''
# ── Cella 13 · Test rapido del modello fine-tunato ────────────────────────────
import torch

model.eval()

# ⚠️  Sostituisci con un prompt reale del tuo dominio
test_prompt = "Explain what is machine learning in simple terms:"

inputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("=" * 60)
print(f"PROMPT   : {test_prompt}")
print("-" * 60)
print(f"RISPOSTA : {response}")
print("=" * 60)
'''

'\n# ── Cella 13 · Test rapido del modello fine-tunato ────────────────────────────\nimport torch\n\nmodel.eval()\n\n# ⚠️  Sostituisci con un prompt reale del tuo dominio\ntest_prompt = "Explain what is machine learning in simple terms:"\n\ninputs = tokenizer(test_prompt, return_tensors="pt").to(model.device)\n\nwith torch.no_grad():\n    outputs = model.generate(\n        **inputs,\n        max_new_tokens=150,\n        temperature=0.7,\n        top_p=0.9,\n        do_sample=True,\n        pad_token_id=tokenizer.pad_token_id,\n    )\n\nresponse = tokenizer.decode(\n    outputs[0][inputs["input_ids"].shape[-1]:],\n    skip_special_tokens=True\n)\n\nprint("=" * 60)\nprint(f"PROMPT   : {test_prompt}")\nprint("-" * 60)\nprint(f"RISPOSTA : {response}")\nprint("=" * 60)\n'